# Advanced Lab 2: Fine-tuning GPT-2 with LoRA

## Objective

In this assignment, you will combine what you have learned in the previous labs to fine-tune a GPT-2 model for sentiment classification using Low-Rank Adaptation (LoRA). Like in the first advanced lab, you are given minimal scaffolding: you will need to design, implement, and evaluate your own solution.

## Background

In labs 2 and 3, you implemented a GPT-2 model from scratch and wrote code to train it with a language modelling objective. In lab 4, you applied LoRA to a pre-trained DistilBERT model for binary sentiment classification on the IMDB movie review dataset. In this assignment, you will adapt the pretrained GPT-2 model for sequence classification and fine-tune it with LoRA.

---

## Instructions

### 1. Set up the model

Load the pre-trained GPT-2 model from lab 3, or the original OpenAI model from lab 2. Replace the language modelling head with a classification head that maps the final hidden state to a binary label.

### 2. Inject LoRA adapters

Implement LoRA adapters and inject them into the attention layers of the GPT-2 model, following the approach from lab 4. Only the LoRA adapter parameters and the classification head should be trainable; all other model parameters should be frozen.

### 3. Fine-tune on IMDB

Fine-tune the adapted model on the same IMDB sentiment dataset used in lab 4. For tokenisation, you can use the `tiktoken` library with the `gpt-2` tokeniser.

### 4. Evaluate and compare

Report the number of parameters of your models and the classification accuracy on the evaluation set. Compare your results against the following baselines from lab 4 in a short summary table:

- DistilBERT (full)
- DistilBERT (LoRA)
- GPT-2 (full) [your work]
- GPT-2 (LoRA) [your work]

### 5. Add your work to your portfolio

Include a short report summarising the design decisions you made, the results you obtained, and your interpretation of the comparison between the models. Add the report and all code to your lab portfolio and present it at the oral exam.

---

## Hints & considerations

- **GPT-2 uses causal (left-to-right) attention**, unlike the bidirectional attention in BERT-family models. Think about what this means for how contextual information is aggregated across tokens when doing classification.
- **The LoRA implementation from Lab 4** wraps a `nn.Linear` layer and can be reused here with minimal or no modification. The target layers and how you identify them will differ between GPT-2 and DistilBERT.
- **GPT-2 uses a larger vocabulary and longer context than DistilBERT.** You may want to truncate input sequences to a manageable length.

---

## Deliverables

- `gpt2-lora.py` — notebook containing your implementation, training run, and evaluation
- `report.md` — short report covering your design decisions, results, and discussion

---

# Implementation

## Imports and Setup

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 178.0 kB/s eta 0:00:00


In [2]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

# For tokenization
import tiktoken

# For data loading and training
import numpy as np
import evaluate
from datasets import load_dataset

---

## GPT-2 Model (from Lab 3)

The following code is copied from `labs/lab3/gpt2.py` without modification.

In [3]:
@dataclass(init=True)
class Config:
    n_vocab: int = 50257
    n_ctx: int = 1024
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12
    num_labels: int = 2


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        (batch_size, seq_len, n_embd) = x.shape
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        return x


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

    def forward(self, x):
        (batch_size, seq_len, n_embd) = x.shape
        head_embd = n_embd // self.n_head
        (q, k, v) = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.n_head, head_embd)
        k = k.view(batch_size, seq_len, self.n_head, head_embd)
        v = v.view(batch_size, seq_len, self.n_head, head_embd)
        q = q.transpose(-2, -3)
        k = k.transpose(-2, -3)
        v = v.transpose(-2, -3)
        x = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # flash attention
        x = x.transpose(-2, -3).contiguous()
        x = x.view(batch_size, seq_len, n_embd)
        x = self.c_proj(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = Attention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


def make_positions(n):
    return torch.arange(n, dtype=torch.long)


class GPT2Model(nn.Module):
    """
    Base GPT-2 model from Lab 3.
    Note: This is the language model version with lm_head.
    You will need to adapt this for classification.
    """
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        (batch_size, seq_len) = x.shape
        wte = self.wte(x)
        wpe = self.wpe(self.pos[:seq_len])
        x = wte + wpe
        x = self.h(x)
        x = self.ln_f(x)
        x = self.lm_head(x)
        return x

---

## Task 1: GPT-2 Model for Sequence Classification

**TODO:** Create a GPT-2 model adapted for binary sentiment classification.

**Key considerations:**
- Replace the language modeling head (`lm_head`) with a classification head
- The classification head should map from `n_embd` to 2 classes (positive/negative)
- Think about which token representation to use for classification:
  - Last token? (GPT-2 is left-to-right, so the last token has seen all previous tokens)
  - First token?
  - Pool all tokens?

**Hints:**
- GPT-2 uses causal attention, so the last token in the sequence has attended to all previous tokens
- You may want to add dropout before the classification layer for regularization

In [4]:
class GPT2ForSequenceClassification(nn.Module):
    """
    GPT-2 model adapted for sequence classification.
    """
    def __init__(self, config, num_labels=2):
        super().__init__()
        self.config = config
        self.num_labels = num_labels

        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.classification_head = nn.Linear(config.n_embd, config.num_labels, bias=True)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)


    def forward(self, token_ids=None, eot_pos=None, labels=None, last_token_idx=None):

        (batch_size, seq_len) = token_ids.shape
        wte = self.wte(token_ids)
        wpe = self.wpe(self.pos[:seq_len])
        x = self.h(wte + wpe)
        x = self.ln_f(x)

        idx = torch.as_tensor(eot_pos, device=token_ids.device, dtype=torch.long)

        x = x[torch.arange(batch_size, device=token_ids.device), idx]
        logits = self.classification_head(x)

        labels = torch.as_tensor(labels).view(-1)
        loss = F.cross_entropy(logits, labels)

        return {"loss": loss, "logits": logits}

---

## Task 2: Load Pre-trained Weights

**TODO:** Load pre-trained GPT-2 weights.

You have two options:
1. Use the model trained in Lab 3 (if available at `../lab3/gpt-2-fineweb-edu.pt`)
2. Load the original OpenAI GPT-2 weights from Hugging Face (as in Lab 2)

**Note:** If using the Lab 3 model, it was trained with `n_vocab=50304` (padded vocab size).

In [5]:
import numpy as np
import torch.nn as nn

def copy_weights(source: np.ndarray, target: torch.Tensor):
    assert source.shape == target.shape
    with torch.no_grad():
        target.copy_(torch.tensor(source, dtype=torch.float32))
def copy_linear(w, b, l: nn.Linear):
    copy_weights(w.transpose(), l.weight.data)
    copy_weights(b, l.bias.data)

def copy_attn(pretrained:dict, idx:int, attn: Attention):
    prefix_key = f'h{idx}.attn'
    attn_key = prefix_key + '.c_attn'
    cproj_key = prefix_key + '.c_proj'
    copy_linear(pretrained[attn_key + '.w'], pretrained[attn_key + '.b'], attn.c_attn)
    copy_linear(pretrained[cproj_key + '.w'], pretrained[cproj_key + '.b'], attn.c_proj)

def copy_ln(pretrained:dict, idx, ln, ln_idx):

    copy_weights(pretrained[f'h{idx}.ln_{ln_idx}.g'], ln.weight.data)
    copy_weights(pretrained[f'h{idx}.ln_{ln_idx}.b'], ln.bias.data)

def copy_mlp(pretrained:dict,idx, mlp):
    prefix_key = f'h{idx}.mlp'
    fc = prefix_key + '.c_fc'
    cproj_key = prefix_key + '.c_proj'
    copy_linear(pretrained[fc + '.w'], pretrained[fc + '.b'], mlp.c_fc)
    copy_linear(pretrained[cproj_key + '.w'], pretrained[cproj_key + '.b'], mlp.c_proj)

def copy_block(pretrained:dict,idx, block:Block):
    copy_ln(pretrained, idx, block.ln_1, 1)
    copy_attn(pretrained, idx, block.attn)
    copy_ln(pretrained, idx, block.ln_2, 2)
    copy_mlp(pretrained, idx, block.mlp)




def load_pretrained_gpt2(model, checkpoint_path=None):

    pretrained = np.load("tdde09-lab/labs/lab2/gpt-2-pretrained.npz")
    # wte
    copy_weights(pretrained['wte'], model.wte.weight.data)
    # wpe
    copy_weights(pretrained['wpe'], model.wpe.weight.data)

    # h
    for i, b in enumerate(model.h):
        copy_block(pretrained, i, b)

    # ln_f
    copy_weights(pretrained['ln_f.b'], model.ln_f.weight.data)
    copy_weights(pretrained['ln_f.g'], model.ln_f.bias.data)

    return model

In [6]:
config = Config()
model = GPT2ForSequenceClassification(config)
model = load_pretrained_gpt2(model)

---

## Task 3: LoRA Adapter Implementation

**TODO:** Implement the LoRA adapter module.

This should be similar to your Lab 4 implementation. The LoRA adapter wraps a pre-trained linear layer and adds a low-rank update.

**Formula:** `y = x @ W_0 + x @ A @ B * (alpha / r)`

Where:
- `W_0` is the frozen pre-trained weight matrix
- `A` and `B` are trainable low-rank matrices
- `r` is the rank
- `alpha` is the scaling hyperparameter

In [7]:
# copied from lab4

class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()
        # TODO: Add your code here
        self.base = pretrained
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.r, self.alpha = rank, alpha
        self.scaling = alpha / rank
        self.A = nn.Parameter(torch.randn(rank, pretrained.in_features))
        self.B = nn.Parameter(torch.zeros(pretrained.out_features, rank))


    def forward(self, x):
        y = self.base(x)
        lora = F.linear(F.linear(x, self.A), self.B) * self.scaling
        return y + lora

---

## Task 4: Inject LoRA Adapters into GPT-2

**TODO:** Implement functions to extract and replace layers, then inject LoRA adapters.

**Key differences from Lab 4:**
- In DistilBERT (Lab 4), LoRA was applied to `q_lin` and `v_lin` layers
- In GPT-2, the attention uses a combined `c_attn` layer for Q, K, V
- You need to decide which layers to target (e.g., `c_attn`, `c_proj`, or both)

**Hints:**
- The `c_attn` layer projects from `n_embd` to `3 * n_embd` (Q, K, V concatenated)
- You may want to apply LoRA only to the Q and V projections within `c_attn`, or apply it to `c_proj`
- Think about how to access the individual Q, K, V weight matrices from the combined `c_attn` layer

In [8]:
def extract(model):
    q_keys = [f'h.{i}.attn.c_attn' for i in range(12)]
    v_keys =  [f'h.{i}.attn.c_proj' for i in range(12)]
    qs = {q_key: model.get_submodule(q_key) for q_key in q_keys}
    vs = {v_key: model.get_submodule(v_key) for v_key in v_keys}

    return {**qs, **vs}


def replace(model, named_layers):
    for key, layer in named_layers.items():
        module = '.'.join(key.split('.')[:-1])
        suffix = key.split('.')[-1]
        setattr(model.get_submodule(module), suffix, layer)

    return model


def inject_lora(model, rank=12):
    """
    Inject LoRA adapters into the model.

    Args:
        model: GPT-2 classification model
        rank: LoRA rank

    Returns:
        Model with LoRA adapters injected
    """
    # Frozen baselayers
    for p in model.parameters():
          p.requires_grad = False
    model.classification_head.requires_grad = True
    # Extract target layers
    layers_to_adapt = extract(model)

    # Wrap each layer with LoRA
    adapted_layers = {}
    for name, layer in layers_to_adapt.items():
        adapted_layers[name] = LoRA(layer, rank=rank, alpha=2*rank)

    # Replace the layers
    replace(model, adapted_layers)

    return model

In [9]:
model = inject_lora(model, 12)

---

## Task 5: Data Loading and Tokenization

**TODO:** Load the IMDB dataset and tokenize it using tiktoken.

**Notes:**
- The IMDB dataset is located at `../lab4/train.csv` and `../lab4/eval.csv`
- Use `tiktoken.get_encoding("gpt2")` for tokenization
- GPT-2 uses BPE tokenization (different from WordPiece used by BERT)
- You may want to truncate sequences to a reasonable length (e.g., 512 tokens)

In [10]:
from datasets import load_dataset
from transformers import AutoTokenizer

enc = tiktoken.get_encoding("gpt2")

def tokenize_function(batch):
    n_ctx = 1024
    token_ids = enc.encode_batch(batch["review"])
    eot_pos = []
    tidsb = []
    for tids in token_ids:
        if len(tids) > n_ctx - 1:
            tids = tids[:(n_ctx-1)]
        eot_pos.append(len(tids))
        tids += [enc.eot_token] * (n_ctx - len(tids))
        tidsb.append(tids)
    return {"token_ids":tidsb, "eot_pos": eot_pos}

def load_and_tokenize_data():
    imdb_dataset = load_dataset(
        "csv", data_files={"train": "tdde09-lab/labs/lab4/train.csv", "eval": "tdde09-lab/labs/lab4/eval.csv"}
    )
    tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)
    return tokenized_imdb_dataset



In [11]:
dataset = load_and_tokenize_data()

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label', 'token_ids', 'eot_pos'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label', 'token_ids', 'eot_pos'],
        num_rows: 500
    })
})

---

## Task 6: Training and Evaluation Functions

**TODO:** Implement training and evaluation functions.

Consider implementing:
1. A function to count trainable parameters
2. A training loop (or use Hugging Face Trainer)
3. An evaluation function to compute accuracy

In [13]:
def num_trainable_parameters(model):
    return sum((p.numel() for p in model.parameters() if p.requires_grad))

num_trainable_parameters(model)

663552

In [14]:
from transformers import Trainer
from transformers import TrainingArguments
import evaluate

training_args = TrainingArguments(
    output_dir="tmp_trainer",
    eval_strategy="epoch",
    logging_steps=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
)

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

def make_trainer(model):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["eval"],
        compute_metrics=compute_metrics,
    )
    return trainer
def train(model):
    print("Number of trainable parameters:", num_trainable_parameters(model))
    trainer = make_trainer(model)
    trainer.train()
    return model
def evaluate(model):
    trainer = make_trainer(model)
    return trainer.evaluate()

In [15]:
train(model)

Number of trainable parameters: 663552


Epoch,Training Loss,Validation Loss,Accuracy
1,0.352601,0.398988,0.880000
2,0.219985,0.441081,0.900000
3,0.504337,0.479881,0.904000


GPT2ForSequenceClassification(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (h): Sequential(
    (0): Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (c_attn): LoRA(
          (base): Linear(in_features=768, out_features=2304, bias=True)
        )
        (c_proj): LoRA(
          (base): Linear(in_features=768, out_features=768, bias=True)
        )
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=768, out_features=3072, bias=True)
        (c_proj): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
    (1): Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (c_attn): LoRA(
          (base): Linear(in_features=768, out_features=2304, bias=True)
        )
        (c_proj): LoRA(
          (base): Linear(in_features=768, out_features=768, bias=True)
        )

In [16]:
evaluate(model)

{'eval_loss': 0.47988077998161316,
 'eval_model_preparation_time': 0.002,
 'eval_accuracy': 0.904,
 'eval_runtime': 36.2674,
 'eval_samples_per_second': 13.786,
 'eval_steps_per_second': 3.447}

---

## Task 7: Full Fine-tuning Baseline

**TODO:** Train the full model (all parameters trainable) as a baseline.

In [17]:
# TODO: Create full fine-tuning model
config = Config()
full_model = GPT2ForSequenceClassification(config)
full_model = load_pretrained_gpt2(full_model)

# TODO: Count parameters
print(f"Trainable parameters: {num_trainable_parameters(full_model)}")

# TODO: Train
train(full_model)

# TODO: Evaluate
# full_accuracy = evaluate_model(full_model, dataset['eval'])
# print(f"Full fine-tuning accuracy: {full_accuracy}")

Trainable parameters: 124441346
Number of trainable parameters: 124441346


Epoch,Training Loss,Validation Loss,Accuracy
1,0.596546,0.299443,0.894000
2,0.354263,0.481595,0.912000
3,0.559679,0.487761,0.906000


GPT2ForSequenceClassification(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (h): Sequential(
    (0): Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (c_attn): Linear(in_features=768, out_features=2304, bias=True)
        (c_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=768, out_features=3072, bias=True)
        (c_proj): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
    (1): Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (c_attn): Linear(in_features=768, out_features=2304, bias=True)
        (c_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=768, o

In [18]:
evaluate(full_model)

{'eval_loss': 0.48776063323020935,
 'eval_model_preparation_time': 0.0016,
 'eval_accuracy': 0.906,
 'eval_runtime': 33.545,
 'eval_samples_per_second': 14.905,
 'eval_steps_per_second': 3.726}

---

## Task 8: LoRA Fine-tuning

**TODO:** Train with LoRA adapters (only LoRA parameters and classification head are trainable).

In [19]:
# TODO: Create LoRA model
# config = Config()
# lora_model = GPT2ForSequenceClassification(config)
# lora_model = load_pretrained_gpt2(lora_model)

# TODO: Inject LoRA adapters
# lora_model = inject_lora(lora_model, rank=12)  # Try different ranks: 4, 8, 12, 16

# TODO: Freeze all non-LoRA parameters except classification head
# (This should already be done in LoRA class, but double-check)

# TODO: Count parameters
# print(f"Trainable parameters: {num_trainable_parameters(lora_model)}")

# TODO: Train
# train_model(lora_model, dataset['train'], dataset['eval'])

# TODO: Evaluate
# lora_accuracy = evaluate_model(lora_model, dataset['eval'])
# print(f"LoRA fine-tuning accuracy: {lora_accuracy}")

---

## Task 9: Results and Comparison

**TODO:** Create a comparison table with your results.

Expected format (fill in your actual results):

| Model | Trainable Parameters | Accuracy |
|-------|---------------------|----------|
| DistilBERT (full) | 66,955,010 | ~90% |
| DistilBERT (LoRA, r=6) | ~300,000 | ~90% |
| GPT-2 (full) | ? | ? |
| GPT-2 (LoRA, r=?) | ? | ? |

**Questions to answer:**
1. How does GPT-2 compare to DistilBERT for this classification task?
2. How much parameter reduction does LoRA achieve for GPT-2?
3. What is the accuracy trade-off between full fine-tuning and LoRA?
4. How does the choice of rank affect the results?
5. What design decisions did you make (e.g., which layers to adapt, which token to use for classification)?

In [20]:
# TODO: Create results table and analysis

# Example structure:
results = {
    "DistilBERT (full)": {
        "parameters": 66955010,
        "accuracy": 0.904,  # Your result from Lab 4
    },
    "DistilBERT (LoRA)": {
        "parameters": 300000,  # Approximate
        "accuracy": 0.904,  # Your result from Lab 4
    },
    "GPT-2 (full)": {
        "parameters": None,  # TODO: Fill in
        "accuracy": None,   # TODO: Fill in
    },
    "GPT-2 (LoRA)": {
        "parameters": None,  # TODO: Fill in
        "accuracy": None,   # TODO: Fill in
    },
}

# Print results table
print("| Model | Trainable Parameters | Accuracy |")
print("|-------|---------------------|----------|")
for model_name, metrics in results.items():
    params = metrics['parameters'] if metrics['parameters'] else 'N/A'
    acc = f"{metrics['accuracy']:.1%}" if metrics['accuracy'] else 'N/A'
    print(f"| {model_name} | {params} | {acc} |")

| Model | Trainable Parameters | Accuracy |
|-------|---------------------|----------|
| DistilBERT (full) | 66955010 | 90.4% |
| DistilBERT (LoRA) | 300000 | 90.4% |
| GPT-2 (full) | N/A | N/A |
| GPT-2 (LoRA) | N/A | N/A |


---

## Bonus: Hyperparameter Exploration

If time permits, experiment with:

1. **Different LoRA ranks** (r=1, 2, 4, 8, 16, 32)
2. **Different alpha values** (e.g., alpha = 2*r, alpha = r)
3. **Different target layers** (c_attn only, c_proj only, or both)
4. **Different classification tokens** (first token, last token, mean pooling)
5. **Different sequence lengths** (256, 512, 1024)

In [21]:
# TODO: Optional hyperparameter exploration

# Example: Try different ranks
# ranks = [1, 2, 4, 8, 16]
# results_by_rank = {}
# for rank in ranks:
#     model = create_lora_model(rank=rank)
#     train_model(model, ...)
#     acc = evaluate_model(model, ...)
#     results_by_rank[rank] = acc
#     print(f"Rank {rank}: {acc:.1%}")

---

**Good luck! **